In [5]:

!pip install -q langchain langchain-ollama llama-index llama-index-llms-ollama llama-index-embeddings-huggingface

import subprocess
import time

# Start Ollama server
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

# Download the model if it is not already installed
!ollama pull llama3.2:1b

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configure LlamaIndex
Settings.llm = Ollama(model="llama3.2:1b", request_timeout=120)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# Create a sample document
text = """
Python is a popular programming language.
LangChain is used to build applications with language models.
LlamaIndex helps search and retrieve information from documents.
A document-based chatbot answers questions using document content.
"""

documents = [Document(text=text)]

# Index the document using LlamaIndex
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

# Create a chatbot using LangChain
llm = ChatOllama(model="llama3.2:1b", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Answer the question using the following document context:
{context}

Question: {question}
Answer:
""")

chain = prompt | llm

# Ask questions
print("Document Q&A Chatbot (type 'exit' to stop)")

while True:
    question = input("\nYou: ")

    if question.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    context = query_engine.query(question)

    response = chain.invoke({
        "context": str(context),
        "question": question
    })

    print("Chatbot:", response.content)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Document Q&A Chatbot (type 'exit' to stop)

You: What is Python?
Chatbot: Python is a high-level, interpreted programming language that focuses on simplicity and readability.

You: What is LangChain?
Chatbot: Based on the provided context, LangChain is a type of language model that is specifically designed to build applications that utilize language understanding and generation capabilities.

You: What does LlamaIndex do?
Chatbot: According to the provided document context, LlamaIndex is a language model that helps retrieve and summarize large amounts of text data into concise and relevant responses.

You: exit
Chatbot: Goodbye!
